In [ ]:
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchvision.transforms.functional as tvF
from mpl_toolkits.axes_grid1 import make_axes_locatable
import math

dtype = torch.float
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

train_args = {
    'alpha': 1e-1,
    'beta': 1e0,
}

# ---------------------------------------------------------
# 1. Model Definitions (TCNN)
# ---------------------------------------------------------
class TCNN(nn.Module):
    def __init__(self, TCNN_args):
        super(TCNN, self).__init__()
        self.Feature_detector = nn.Sequential(
            nn.Conv1d(in_channels=TCNN_args['in_channels'], out_channels=TCNN_args['out_channels'], kernel_size=TCNN_args['kernel_size']),
            nn.ReLU(),
        )
        self.Norm_Pool = nn.Sequential(
            nn.BatchNorm1d(num_features=TCNN_args['out_channels'], affine=False),
            nn.MaxPool1d(kernel_size=TCNN_args['maxpool_size']),
        )
        self.FC = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Linear(in_features=TCNN_args['out_channels'] * int((TCNN_args['batch_size'] - TCNN_args['kernel_size'] + 1 - 2) / TCNN_args['maxpool_size'] + 1), out_features=TCNN_args['category_num'], bias=False),
        )

    def forward(self, x):
        return F.log_softmax(self.FC(x), dim=1)

def kernel_init_tcnn(run_TCNN, TCNN_args):
    for param in run_TCNN.Feature_detector.parameters():
        param.requires_grad = False
    run_TCNN_param = run_TCNN.state_dict()
    step = TCNN_args['kernel_size'] // 6 // TCNN_args['out_channels']
    for ii in range(0, TCNN_args['out_channels']):
        run_TCNN_param['Feature_detector.0.weight'][ii] = torch.tensor(signal.windows.gaussian(TCNN_args['kernel_size'], 2 + ii * step), dtype=dtype).to(device)
    run_TCNN_param['Feature_detector.0.bias'][:] = 0.0
    run_TCNN.load_state_dict(run_TCNN_param)
    return run_TCNN


# ---------------------------------------------------------
# 2. Model Definitions (TTE)
# ---------------------------------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, num_hiddens, dropout, max_len=1000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        self.P = torch.zeros((1, max_len, num_hiddens))
        X = torch.arange(max_len, dtype=torch.float32).reshape(-1, 1) / torch.pow(10000, torch.arange(0, num_hiddens, 2, dtype=torch.float32) / num_hiddens)
        self.P[:, :, 0::2] = torch.sin(X)
        self.P[:, :, 1::2] = torch.cos(X[:,0:2])

    def forward(self, X):
        X = X + self.P[:, :X.shape[1], :].to(X.device)
        return self.dropout(X)

class ConvEmbeding(nn.Module):
    def __init__(self, TTE_args):
        super(ConvEmbeding, self).__init__()
        self.Feature_detector = nn.Sequential(
            nn.Conv1d(in_channels=TTE_args['input_channels'], out_channels=TTE_args['expected_dim'],
                    kernel_size=TTE_args['kernel_size'], stride=TTE_args['stride']),
            nn.ReLU(),
        )
        self.Norm_Pool = nn.Sequential(
            nn.BatchNorm1d(num_features=TTE_args['expected_dim'], affine=False),
            nn.MaxPool1d(kernel_size=TTE_args['maxpool_size']),
        )
        self.Embeding = nn.Sequential(
            self.Feature_detector,
            self.Norm_Pool,
        )

    def forward(self, x):
        return self.Embeding(x).permute(0,2,1)

class TemporalTransformerEncoder(nn.Module):
    def __init__(self, TTE_args):
        super(TemporalTransformerEncoder, self).__init__()
        self.Embeding = ConvEmbeding(TTE_args)
        self.PositionalEncoding = PositionalEncoding(TTE_args['expected_dim'], TTE_args['position_dropout'])
        self.transformer_encoder = nn.TransformerEncoder(nn.TransformerEncoderLayer(TTE_args['expected_dim'], TTE_args['num_heads'], batch_first=True), TTE_args['num_layers'])
        self.FC = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.LazyLinear(out_features=TTE_args['category_num']),
        )

    def forward(self, x):
        return F.log_softmax(self.FC(self.transformer_encoder(self.PositionalEncoding(x))), dim=1)

def kernel_init_tte(run_TTE, TTE_args):
    for param in run_TTE.Embeding.Feature_detector.parameters():
        param.requires_grad = False
    run_TTE_param = run_TTE.state_dict()
    step = TTE_args['kernel_size'] // 6 // TTE_args['expected_dim']
    for ii in range(0, TTE_args['expected_dim']):
        run_TTE_param['Embeding.Feature_detector.0.weight'][ii] = torch.tensor(signal.windows.gaussian(TTE_args['kernel_size'], 2 + ii * step), dtype=dtype).to(device)
    run_TTE_param['Embeding.Feature_detector.0.bias'][:] = 0.0
    run_TTE.load_state_dict(run_TTE_param)
    return run_TTE


# ---------------------------------------------------------
# 3. Helper Functions
# ---------------------------------------------------------
def confusion_cal(y_test, py_test, y_type):
    class_num = len(y_test.unique())
    confusion_num = np.zeros((class_num, class_num))
    if y_type == 0:
        for ii in range(0, class_num):
            for jj in range(0, class_num):
                confusion_num[ii,jj] = len(np.intersect1d(torch.argwhere(torch.argmax(py_test, dim=1) == jj).detach().cpu().numpy(), torch.argwhere(y_test == ii).detach().cpu().numpy()))
    elif y_type == 1:
        for ii in range(0, class_num):
            for jj in range(0, class_num):
                confusion_num[ii,jj] = len(np.intersect1d(torch.argwhere(py_test == jj).detach().cpu().numpy(), torch.argwhere(y_test == ii).detach().cpu().numpy()))
    return confusion_num

def acu_cal(log_py_train, y_train, log_py_test, y_test, acu):
    acu['acu_train'].append((torch.sum(torch.argmax(log_py_train, dim=1) == y_train) / y_train.size(0)).item())
    acu['acu_test'].append((torch.sum(torch.argmax(log_py_test, dim=1) == y_test) / y_test.size(0)).item())
    acu['acu_train_STDP'].append((torch.sum(torch.argmax(log_py_train, dim=1)[torch.argwhere(y_train == 0)] == y_train[torch.argwhere(y_train == 0)]) / y_train[torch.argwhere(y_train == 0)].size(0)).item())
    acu['acu_test_STDP'].append((torch.sum(torch.argmax(log_py_test, dim=1)[torch.argwhere(y_test == 0)] == y_test[torch.argwhere(y_test == 0)]) / y_test[torch.argwhere(y_test == 0)].size(0)).item())
    acu['acu_train_BurstProp'].append((torch.sum(torch.argmax(log_py_train, dim=1)[torch.argwhere(y_train == 1)] == y_train[torch.argwhere(y_train == 1)]) / y_train[torch.argwhere(y_train == 1)].size(0)).item())
    acu['acu_test_BurstProp'].append((torch.sum(torch.argmax(log_py_test, dim=1)[torch.argwhere(y_test == 1)] == y_test[torch.argwhere(y_test == 1)]) / y_test[torch.argwhere(y_test == 1)].size(0)).item())
    acu['acu_train_BP1'].append((torch.sum(torch.argmax(log_py_train, dim=1)[torch.argwhere(y_train == 2)] == y_train[torch.argwhere(y_train == 2)]) / y_train[torch.argwhere(y_train == 2)].size(0)).item())
    acu['acu_test_BP1'].append((torch.sum(torch.argmax(log_py_test, dim=1)[torch.argwhere(y_test == 2)] == y_test[torch.argwhere(y_test == 2)]) / y_test[torch.argwhere(y_test == 2)].size(0)).item())
    acu['acu_train_BP2'].append((torch.sum(torch.argmax(log_py_train, dim=1)[torch.argwhere(y_train == 3)] == y_train[torch.argwhere(y_train == 3)]) / y_train[torch.argwhere(y_train == 3)].size(0)).item())
    acu['acu_test_BP2'].append((torch.sum(torch.argmax(log_py_test, dim=1)[torch.argwhere(y_test == 3)] == y_test[torch.argwhere(y_test == 3)]) / y_test[torch.argwhere(y_test == 3)].size(0)).item())
    acu['acu_train_NC'].append((torch.sum(torch.argmax(log_py_train, dim=1)[torch.argwhere(y_train == 4)] == y_train[torch.argwhere(y_train == 4)]) / y_train[torch.argwhere(y_train == 4)].size(0)).item())
    acu['acu_test_NC'].append((torch.sum(torch.argmax(log_py_test, dim=1)[torch.argwhere(y_test == 4)] == y_test[torch.argwhere(y_test == 4)]) / y_test[torch.argwhere(y_test == 4)].size(0)).item())
    return acu

def init_acu_dict():
    return {'acu_train': [], 'acu_test': [], 'acu_train_STDP': [], 'acu_test_STDP': [],
            'acu_train_BurstProp': [], 'acu_test_BurstProp': [], 'acu_train_BP1': [], 'acu_test_BP1': [],
            'acu_train_BP2': [], 'acu_test_BP2': [], 'acu_train_NC': [], 'acu_test_NC': []}


# ---------------------------------------------------------
# 4. Data Loading
# ---------------------------------------------------------
dat_dir = '/content/drive/MyDrive/Project/PlasticityDecoding/data'

print("Loading Data...")
SPK = []
ptype = []
for rnum in range(0, 30):
    SPK.append(np.load(dat_dir + f'/FixedRepresentation/Spk_an_wn_{rnum}.npy'))
    ptype.append(np.load(dat_dir + f'/FixedRepresentation/Ptype_an_wn_{rnum}.npy'))

SPK = np.concatenate(SPK, axis=0)
ptype = np.concatenate(ptype, axis=0)

SPK1 = []
ptype1 = []
for rnum in range(0, 49):
    SPK1.append(np.load(dat_dir + f'/FixedRepresentation01/Spk_an_wn_{rnum}.npy'))
    ptype1.append(np.load(dat_dir + f'/FixedRepresentation01/Ptype_an_wn_{rnum}.npy'))

SPK1 = np.concatenate(SPK1, axis=0)
ptype1 = np.concatenate(ptype1, axis=0)

# Calculate temporal split
half_time = SPK.shape[1] // 2

# Test 1 Setup (SPK Only: Burst Fraction = 0.2)
x_train_1 = torch.tensor(SPK[:, :half_time], dtype=dtype).to(device).unsqueeze(dim=1)
y_train_1 = torch.tensor(ptype, dtype=torch.long).to(device)
x_test_1 = torch.tensor(SPK[:, half_time:], dtype=dtype).to(device).unsqueeze(dim=1)
y_test_1 = torch.tensor(ptype, dtype=torch.long).to(device)

# Test 2 Setup (Merged SPK + SPK1: Burst Fraction = 0.2 and 0.4)
SPK_merged = np.concatenate([SPK, SPK1], axis=0)
ptype_merged = np.concatenate([ptype, ptype1], axis=0)

x_train_2 = torch.tensor(SPK_merged[:, :half_time], dtype=dtype).to(device).unsqueeze(dim=1)
y_train_2 = torch.tensor(ptype_merged, dtype=torch.long).to(device)
x_test_2 = torch.tensor(SPK_merged[:, half_time:], dtype=dtype).to(device).unsqueeze(dim=1)
y_test_2 = torch.tensor(ptype_merged, dtype=torch.long).to(device)

# ---------------------------------------------------------
# 5. Training Logic
# ---------------------------------------------------------
# Result Arrays (50 runs, 12 metrics each)
tcnn_acu_test1 = np.zeros((50, 12))
tcnn_acu_test2 = np.zeros((50, 12))
tte_acu_test1  = np.zeros((50, 12))
tte_acu_test2  = np.zeros((50, 12))

# Network Arguments
TCNN_args_1 = {
    'in_channels': 1, 'out_channels': 5, 'kernel_size': 500, 'maxpool_size': 125,
    'category_num': 5, 'batch_size': x_train_1.size(axis=2),
}
TCNN_args_2 = {
    'in_channels': 1, 'out_channels': 5, 'kernel_size': 500, 'maxpool_size': 125,
    'category_num': 5, 'batch_size': x_train_2.size(axis=2),
}
TTE_args_1 = {
    'input_channels': 1, 'kernel_size': 500, 'stride': 1, 'maxpool_size': 125,
    'num_heads': 5, 'expected_dim': 5, 'num_layers': 1, 'category_num': 5,
    'position_dropout': 0.0, 'data_len': x_train_1.size(dim=1),
}
TTE_args_2 = {
    'input_channels': 1, 'kernel_size': 500, 'stride': 1, 'maxpool_size': 125,
    'num_heads': 5, 'expected_dim': 5, 'num_layers': 1, 'category_num': 5,
    'position_dropout': 0.0, 'data_len': x_train_2.size(dim=1),
}

loss_fn = nn.NLLLoss()

print("Starting Training Loops (50 iterations)...")
for ii in range(0, 50):

    # ------------------ TCNN TEST 1 (SPK) ------------------
    run_TCNN = TCNN(TCNN_args_1).to(device)
    run_TCNN = kernel_init_tcnn(run_TCNN, TCNN_args_1)
    optimizer = torch.optim.Adam(run_TCNN.parameters(), lr=1e-3)

    run_TCNN.train()
    xt1_pool = run_TCNN.Norm_Pool(run_TCNN.Feature_detector(x_train_1))
    xt1_test_pool = run_TCNN.Norm_Pool(run_TCNN.Feature_detector(x_test_1))

    for epoch in range(0, 500):
        log_py_train = run_TCNN(xt1_pool)
        for weight in run_TCNN.FC.parameters():
            L1 = train_args['beta'] * F.l1_loss(torch.zeros_like(weight), weight)
        L = train_args['alpha'] * loss_fn(log_py_train, y_train_1) + L1
        optimizer.zero_grad()
        L.backward()
        optimizer.step()
        if torch.sum(torch.argmax(log_py_train, dim=1) == y_train_1) / y_train_1.size(0) > 0.85:
            break

    run_TCNN.eval()
    log_py_test = run_TCNN(xt1_test_pool)
    acu_tcnn_1 = acu_cal(log_py_train, y_train_1, log_py_test, y_test_1, init_acu_dict())

    # Map metrics for Test 1
    tcnn_acu_test1[ii,0] = acu_tcnn_1['acu_train'][0]
    tcnn_acu_test1[ii,1] = acu_tcnn_1['acu_test'][0]
    tcnn_acu_test1[ii,2] = acu_tcnn_1['acu_train_STDP'][0]
    tcnn_acu_test1[ii,3] = acu_tcnn_1['acu_test_STDP'][0]
    tcnn_acu_test1[ii,4] = acu_tcnn_1['acu_train_BurstProp'][0]
    tcnn_acu_test1[ii,5] = acu_tcnn_1['acu_test_BurstProp'][0]
    tcnn_acu_test1[ii,6] = acu_tcnn_1['acu_train_BP1'][0]
    tcnn_acu_test1[ii,7] = acu_tcnn_1['acu_test_BP1'][0]
    tcnn_acu_test1[ii,8] = acu_tcnn_1['acu_train_BP2'][0]
    tcnn_acu_test1[ii,9] = acu_tcnn_1['acu_test_BP2'][0]
    tcnn_acu_test1[ii,10] = acu_tcnn_1['acu_train_NC'][0]
    tcnn_acu_test1[ii,11] = acu_tcnn_1['acu_test_NC'][0]

    # ------------------ TCNN TEST 2 (Merged) ------------------
    run_TCNN_2 = TCNN(TCNN_args_2).to(device)
    run_TCNN_2 = kernel_init_tcnn(run_TCNN_2, TCNN_args_2)
    optimizer_2 = torch.optim.Adam(run_TCNN_2.parameters(), lr=1e-3)

    run_TCNN_2.train()
    xt2_pool = run_TCNN_2.Norm_Pool(run_TCNN_2.Feature_detector(x_train_2))
    xt2_test_pool = run_TCNN_2.Norm_Pool(run_TCNN_2.Feature_detector(x_test_2))

    for epoch in range(0, 500):
        log_py_train = run_TCNN_2(xt2_pool)
        for weight in run_TCNN_2.FC.parameters():
            L1 = train_args['beta'] * F.l1_loss(torch.zeros_like(weight), weight)
        L = train_args['alpha'] * loss_fn(log_py_train, y_train_2) + L1
        optimizer_2.zero_grad()
        L.backward()
        optimizer_2.step()
        if torch.sum(torch.argmax(log_py_train, dim=1) == y_train_2) / y_train_2.size(0) > 0.85:
            break

    run_TCNN_2.eval()
    log_py_test_2 = run_TCNN_2(xt2_test_pool)
    acu_tcnn_2 = acu_cal(log_py_train, y_train_2, log_py_test_2, y_test_2, init_acu_dict())

    tcnn_acu_test2[ii,0] = acu_tcnn_2['acu_train'][0]
    tcnn_acu_test2[ii,1] = acu_tcnn_2['acu_test'][0]
    tcnn_acu_test2[ii,2] = acu_tcnn_2['acu_train_STDP'][0]
    tcnn_acu_test2[ii,3] = acu_tcnn_2['acu_test_STDP'][0]
    tcnn_acu_test2[ii,4] = acu_tcnn_2['acu_train_BurstProp'][0]
    tcnn_acu_test2[ii,5] = acu_tcnn_2['acu_test_BurstProp'][0]
    tcnn_acu_test2[ii,6] = acu_tcnn_2['acu_train_BP1'][0]
    tcnn_acu_test2[ii,7] = acu_tcnn_2['acu_test_BP1'][0]
    tcnn_acu_test2[ii,8] = acu_tcnn_2['acu_train_BP2'][0]
    tcnn_acu_test2[ii,9] = acu_tcnn_2['acu_test_BP2'][0]
    tcnn_acu_test2[ii,10] = acu_tcnn_2['acu_train_NC'][0]
    tcnn_acu_test2[ii,11] = acu_tcnn_2['acu_test_NC'][0]

    # ------------------ TTE TEST 1 (SPK) ------------------
    run_TTE = TemporalTransformerEncoder(TTE_args_1).to(device)
    run_TTE = kernel_init_tte(run_TTE, TTE_args_1)
    optimizer_tte = torch.optim.Adam(run_TTE.parameters(), lr=1e-3)

    run_TTE.train()
    xtte1_embed = run_TTE.Embeding(x_train_1)
    xtte1_test_embed = run_TTE.Embeding(x_test_1)

    for epoch in range(0, 500):
        log_py_train = run_TTE(xtte1_embed)
        L = loss_fn(log_py_train, y_train_1)
        optimizer_tte.zero_grad()
        L.backward()
        optimizer_tte.step()
        if torch.sum(torch.argmax(log_py_train, dim=1) == y_train_1) / y_train_1.size(0) > 0.85:
            break

    run_TTE.eval()
    log_py_test_tte = run_TTE(xtte1_test_embed)
    acu_tte_1 = acu_cal(log_py_train, y_train_1, log_py_test_tte, y_test_1, init_acu_dict())

    tte_acu_test1[ii,0] = acu_tte_1['acu_train'][0]
    tte_acu_test1[ii,1] = acu_tte_1['acu_test'][0]
    tte_acu_test1[ii,2] = acu_tte_1['acu_train_STDP'][0]
    tte_acu_test1[ii,3] = acu_tte_1['acu_test_STDP'][0]
    tte_acu_test1[ii,4] = acu_tte_1['acu_train_BurstProp'][0]
    tte_acu_test1[ii,5] = acu_tte_1['acu_test_BurstProp'][0]
    tte_acu_test1[ii,6] = acu_tte_1['acu_train_BP1'][0]
    tte_acu_test1[ii,7] = acu_tte_1['acu_test_BP1'][0]
    tte_acu_test1[ii,8] = acu_tte_1['acu_train_BP2'][0]
    tte_acu_test1[ii,9] = acu_tte_1['acu_test_BP2'][0]
    tte_acu_test1[ii,10] = acu_tte_1['acu_train_NC'][0]
    tte_acu_test1[ii,11] = acu_tte_1['acu_test_NC'][0]

    # ------------------ TTE TEST 2 (Merged) ------------------
    run_TTE_2 = TemporalTransformerEncoder(TTE_args_2).to(device)
    run_TTE_2 = kernel_init_tte(run_TTE_2, TTE_args_2)
    optimizer_tte_2 = torch.optim.Adam(run_TTE_2.parameters(), lr=1e-3)

    run_TTE_2.train()
    xtte2_embed = run_TTE_2.Embeding(x_train_2)
    xtte2_test_embed = run_TTE_2.Embeding(x_test_2)

    for epoch in range(0, 500):
        log_py_train = run_TTE_2(xtte2_embed)
        L = loss_fn(log_py_train, y_train_2)
        optimizer_tte_2.zero_grad()
        L.backward()
        optimizer_tte_2.step()
        if torch.sum(torch.argmax(log_py_train, dim=1) == y_train_2) / y_train_2.size(0) > 0.85:
            break

    run_TTE_2.eval()
    log_py_test_tte_2 = run_TTE_2(xtte2_test_embed)
    acu_tte_2 = acu_cal(log_py_train, y_train_2, log_py_test_tte_2, y_test_2, init_acu_dict())

    tte_acu_test2[ii,0] = acu_tte_2['acu_train'][0]
    tte_acu_test2[ii,1] = acu_tte_2['acu_test'][0]
    tte_acu_test2[ii,2] = acu_tte_2['acu_train_STDP'][0]
    tte_acu_test2[ii,3] = acu_tte_2['acu_test_STDP'][0]
    tte_acu_test2[ii,4] = acu_tte_2['acu_train_BurstProp'][0]
    tte_acu_test2[ii,5] = acu_tte_2['acu_test_BurstProp'][0]
    tte_acu_test2[ii,6] = acu_tte_2['acu_train_BP1'][0]
    tte_acu_test2[ii,7] = acu_tte_2['acu_test_BP1'][0]
    tte_acu_test2[ii,8] = acu_tte_2['acu_train_BP2'][0]
    tte_acu_test2[ii,9] = acu_tte_2['acu_test_BP2'][0]
    tte_acu_test2[ii,10] = acu_tte_2['acu_train_NC'][0]
    tte_acu_test2[ii,11] = acu_tte_2['acu_test_NC'][0]

# ---------------------------------------------------------
# 6. Saving and Output
# ---------------------------------------------------------
save_dir = '/content/drive/MyDrive/Project/PlasticityDecoding/result_20260601'
os.makedirs(save_dir, exist_ok=True)

np.save(save_dir + '/TCNN_OOD_time_SPK.npy', tcnn_acu_test1)
np.save(save_dir + '/TCNN_OOD_time_Merged.npy', tcnn_acu_test2)
np.save(save_dir + '/TTE_OOD_time_SPK.npy', tte_acu_test1)
np.save(save_dir + '/TTE_OOD_time_Merged.npy', tte_acu_test2)

tcnn_mean_1 = np.round(tcnn_acu_test1.mean(axis=0), 3)
tcnn_mean_2 = np.round(tcnn_acu_test2.mean(axis=0), 3)
tte_mean_1 = np.round(tte_acu_test1.mean(axis=0), 3)
tte_mean_2 = np.round(tte_acu_test2.mean(axis=0), 3)

print("\n==============================")
print("Results: Temporal Generalization (Early Half -> Late Half)")
print("==============================")
print("[Test 1: SPK Only (Burst Fraction = 0.2)]")
print(f"TCNN Overall (Train / Test) : {tcnn_mean_1[0]:.3f} / {tcnn_mean_1[1]:.3f}")
print(f"  - STDP                    : {tcnn_mean_1[2]:.3f} / {tcnn_mean_1[3]:.3f}")
print(f"  - BurstProp               : {tcnn_mean_1[4]:.3f} / {tcnn_mean_1[5]:.3f}")
print(f"  - BPTT1                   : {tcnn_mean_1[6]:.3f} / {tcnn_mean_1[7]:.3f}")
print(f"  - BPTT2                   : {tcnn_mean_1[8]:.3f} / {tcnn_mean_1[9]:.3f}")
print(f"  - NC                      : {tcnn_mean_1[10]:.3f} / {tcnn_mean_1[11]:.3f}")
print(f"TTE  Overall (Train / Test) : {tte_mean_1[0]:.3f} / {tte_mean_1[1]:.3f}")
print(f"  - STDP                    : {tte_mean_1[2]:.3f} / {tte_mean_1[3]:.3f}")
print(f"  - BurstProp               : {tte_mean_1[4]:.3f} / {tte_mean_1[5]:.3f}")
print(f"  - BPTT1                   : {tte_mean_1[6]:.3f} / {tte_mean_1[7]:.3f}")
print(f"  - BPTT2                   : {tte_mean_1[8]:.3f} / {tte_mean_1[9]:.3f}")
print(f"  - NC                      : {tte_mean_1[10]:.3f} / {tte_mean_1[11]:.3f}")

print("\n[Test 2: Merged SPK + SPK1 (Burst Frc = 0.2 & 0.4)]")
print(f"TCNN Overall (Train / Test) : {tcnn_mean_2[0]:.3f} / {tcnn_mean_2[1]:.3f}")
print(f"  - STDP                    : {tcnn_mean_2[2]:.3f} / {tcnn_mean_2[3]:.3f}")
print(f"  - BurstProp               : {tcnn_mean_2[4]:.3f} / {tcnn_mean_2[5]:.3f}")
print(f"  - BPTT1                   : {tcnn_mean_2[6]:.3f} / {tcnn_mean_2[7]:.3f}")
print(f"  - BPTT2                   : {tcnn_mean_2[8]:.3f} / {tcnn_mean_2[9]:.3f}")
print(f"  - NC                      : {tcnn_mean_2[10]:.3f} / {tcnn_mean_2[11]:.3f}")
print(f"TTE  Overall (Train / Test) : {tte_mean_2[0]:.3f} / {tte_mean_2[1]:.3f}")
print(f"  - STDP                    : {tte_mean_2[2]:.3f} / {tte_mean_2[3]:.3f}")
print(f"  - BurstProp               : {tte_mean_2[4]:.3f} / {tte_mean_2[5]:.3f}")
print(f"  - BPTT1                   : {tte_mean_2[6]:.3f} / {tte_mean_2[7]:.3f}")
print(f"  - BPTT2                   : {tte_mean_2[8]:.3f} / {tte_mean_2[9]:.3f}")
print(f"  - NC                      : {tte_mean_2[10]:.3f} / {tte_mean_2[11]:.3f}\n")
print("Done! Files saved.")

Loading Data...
Starting Training Loops (50 iterations)...


/tmp/ipykernel_788/4238886193.py:101: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer_encoder = nn.TransformerEncoder(nn.TransformerEncoderLayer(TTE_args['expected_dim'], TTE_args['num_heads'], batch_first=True), TTE_args['num_layers'])



Results: Temporal Generalization (Early Half -> Late Half)
[Test 1: SPK Only (Burst Fraction = 0.2)]
TCNN Overall (Train / Test) : 0.667 / 0.406
  - STDP                    : 0.761 / 0.013
  - BurstProp               : 0.414 / 0.193
  - BPTT1                   : 0.532 / 0.089
  - BPTT2                   : 0.773 / 0.827
  - NC                      : 0.855 / 0.906
TTE  Overall (Train / Test) : 0.752 / 0.267
  - STDP                    : 0.797 / 0.009
  - BurstProp               : 0.667 / 0.085
  - BPTT1                   : 0.634 / 0.093
  - BPTT2                   : 0.786 / 0.870
  - NC                      : 0.876 / 0.277

[Test 2: Merged SPK + SPK1 (Burst Frc = 0.2 & 0.4)]
TCNN Overall (Train / Test) : 0.620 / 0.427
  - STDP                    : 0.782 / 0.020
  - BurstProp               : 0.283 / 0.325
  - BPTT1                   : 0.502 / 0.081
  - BPTT2                   : 0.705 / 0.822
  - NC                      : 0.831 / 0.888
TTE  Overall (Train / Test) : 0.698 / 0.259
  - STDP 